# 01. Минимальный Deep Agent + filesystem

Начинаем с API shape `messages`, затем добавляем Deep Agent и файловый backend. Это первый runnable слой OpenClaw-клона.


![Build path](../visuals/openclaw_path/04_build_path.svg)

В этом notebook делаем первые два шага: minimal agent и workspace-scoped filesystem.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)

def write_text(path: str, content: str) -> Path:
    target = REPO_ROOT / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return target

def write_json(path: str, payload: dict) -> Path:
    target = REPO_ROOT / path
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n')
    return target


In [ ]:
BASE_PROMPT = """\
You are OpenClaw, a workshop clone of a messenger-first Deep Agent runtime.
You help with software engineering and DevOps tasks through safe tool use.

Rules:
- start from the user's messages and preserve context;
- inspect before acting;
- keep external actions in dry-run mode first;
- never print full secrets;
- explain the result briefly and concretely.
"""

messages_input = {
    "messages": [
        {"role": "user", "content": "Коротко объясни, как ты будешь проверять незнакомый репозиторий."}
    ]
}

minimal_agent = create_deep_agent(
    model=model_name(),
    tools=[],
    system_prompt=BASE_PROMPT,
)

print(type(minimal_agent).__name__)
print(messages_input)


## Добавляем filesystem backend

Filesystem — первый слой, где агент перестаёт быть «ответчиком» и получает рабочую среду. Shell оставляем за отдельным env-gate.


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\n\nload_dotenv()\n\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\n\ndef _model() -> str:\n    return os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL)\n\n\ndef _workspace_root() -> Path:\n    return Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).expanduser().resolve()\n\n\ndef _backend():\n    root_dir = _workspace_root()\n    if os.getenv(\"OPENCLAW_ENABLE_LOCAL_SHELL\") == \"1\":\n        from deepagents.backends import LocalShellBackend\n\n        return LocalShellBackend(\n            root_dir=root_dir,\n            virtual_mode=True,\n            inherit_env=True,\n            timeout=120,\n            max_output_bytes=80_000,\n        )\n\n    from deepagents.backends import FilesystemBackend\n\n    return FilesystemBackend(root_dir=root_dir, virtual_mode=True)\n\n\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw, a workshop clone of a messenger-first Deep Agent runtime.\nStart from messages, inspect the workspace before changing files, and keep shell\naccess behind OPENCLAW_ENABLE_LOCAL_SHELL=1.\n\"\"\"\n\nagent = create_deep_agent(\n    model=_model(),\n    tools=[],\n    system_prompt=BASE_PROMPT,\n    backend=_backend(),\n)\n"

CONFIG = {
    "dependencies": ["."],
    "graphs": {"openclaw_path_01": "./agents/openclaw_path_01_filesystem.py:agent"},
    "env": ".env",
}

entrypoint = write_text('agents/openclaw_path_01_filesystem.py', ENTRYPOINT)
config = write_json('langgraph.openclaw_path.json', CONFIG)
print(entrypoint.relative_to(REPO_ROOT))
print(config.relative_to(REPO_ROOT))


## Проверочный prompt

```text
Прочитай файлы в корне проекта и скажи, из каких частей состоит этот OpenClaw workshop.
```

Запуск:

```bash
uv run langgraph dev --config langgraph.openclaw_path.json
```
